> **Notebook Goal**\n> Train a KNN income classifier, measure global individual fairness (IF), then report within-group IF for marital-status, sex, and their intersection.


> **Imports**\n> Load pandas and scikit-learn components for preprocessing, KNN training, and evaluation metrics.


In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


> **Targets and Sensitive Attributes**\n> Define the prediction target and list sensitive attributes. Sensitive attributes are excluded from training features and kept for fairness analysis.


In [2]:
target = "income-class"
sensitive_cols = ["marital-status", "relationship", "race", "sex", "native-country"]

> **Load and Split Data**\n> Read the Adult dataset, separate train/test sets, and preserve sensitive columns for later subgroup analysis.


In [3]:
data = pd.read_csv("adult.data.csv", na_values="?", skipinitialspace=True)

feature_cols = [c for c in data.columns if c not in [target] + sensitive_cols]
X = data[feature_cols]
y = data[target]
X_sensitive = data[sensitive_cols]

X_train, X_test, y_train, y_test, _, X_sensitivefeatures_test = train_test_split(
    X, y, X_sensitive, test_size=0.20, random_state=42, stratify=y
)

> **Preprocessing Pipeline**\n> Create numeric and categorical preprocessing steps (impute + scale / one-hot) used by the KNN pipeline.


In [4]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

> **Train Baseline KNN**\n> Fit the selected KNN configuration and print standard classification metrics before fairness analysis.


In [5]:
classifier = KNeighborsClassifier(
    n_neighbors=17,
    weights="uniform",
    leaf_size=20,
    p=2,
)

classifierPipeline = Pipeline([
    ("preprocess", preprocess),
    ("knn", classifier),
])

classifierPipeline.fit(X_train, y_train)
y_predict = classifierPipeline.predict(X_test)  # <- use pipeline here

print(confusion_matrix(y_test, y_predict))
print(classification_report(y_test, y_predict))

[[4615  330]
 [ 860  708]]
              precision    recall  f1-score   support

       <=50K       0.84      0.93      0.89      4945
        >50K       0.68      0.45      0.54      1568

    accuracy                           0.82      6513
   macro avg       0.76      0.69      0.71      6513
weighted avg       0.80      0.82      0.80      6513



## Individual Fairness Analysis (Minimal)
Concise IF audit using local nearest-neighbor pairs, probability outputs, and a normalized distance metric in preprocessed non-sensitive feature space.


> **Global IF (Local Pair Audit)**\n> Compute local pairs, normalize distances, calculate violations `max(0, |p_i-p_j| - d_ij)`, and print the global IF summary.


In [6]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

# Probability output p(x) for the positive class
p = classifierPipeline.predict_proba(X_test)[:, list(classifierPipeline.classes_).index(">50K")]

# Similarity metric d(x, y): normalized distance in preprocessed non-sensitive feature space
X_test_preprocessed = classifierPipeline.named_steps["preprocess"].transform(X_test)

# Local pairs: fixed number of nearest neighbors (excluding self)
k_local = 15
nn = NearestNeighbors(n_neighbors=min(k_local + 1, len(X_test)), metric="euclidean")
nn.fit(X_test_preprocessed)
distances, indices = nn.kneighbors(X_test_preprocessed)

rows = np.repeat(np.arange(len(X_test)), distances.shape[1] - 1)
cols = indices[:, 1:].reshape(-1)
raw_d = distances[:, 1:].reshape(-1)

# Normalize distances into [0, 1]
scale = raw_d.max()
if scale == 0:
    scale = 1.0
d = np.clip(raw_d / scale, 0.0, 1.0)

# Fairness violations: max(0, |p_i - p_j| - d_ij)
delta_p = np.abs(p[rows] - p[cols])
violations = np.maximum(0.0, delta_p - d)

print("Individual fairness (local) summary")
print("- evaluated pairs:", len(violations))
print("- max_violation:", round(float(violations.max()), 6))
print("- avg_violation:", round(float(violations.mean()), 6))
print("- violation_rate:", round(float((violations > 0).mean()), 6))


Individual fairness (local) summary
- evaluated pairs: 97695
- max_violation: 0.773675
- avg_violation: 0.022848
- violation_rate: 0.260648


> **Subgroup and Intersection IF**\n> Reuse the pair-level fairness outputs to report within-group IF for marital-status, sex, and the marital-status x sex intersection.


In [7]:
# Within-group IF for selected analysis attributes and their intersection
required = ["rows", "cols", "violations", "X_sensitivefeatures_test"]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError("Run the local fairness cell first. Missing: " + ", ".join(missing))

sensitive_test = X_sensitivefeatures_test.reset_index(drop=True).copy()
analysis_attrs = ["marital-status", "sex"]


def within_group_if_table_from_labels(labels, attribute_name):
    labels = labels.fillna("<missing>").astype(str)
    labels_arr = labels.to_numpy()
    group_i = labels_arr[rows]
    group_j = labels_arr[cols]
    within_mask = group_i == group_j
    sample_counts = labels.value_counts().to_dict()

    records = []
    for subgroup in sorted(sample_counts):
        mask = within_mask & (group_i == subgroup)
        n_pairs = int(mask.sum())
        if n_pairs == 0:
            records.append({
                "attribute": attribute_name,
                "subgroup": subgroup,
                "n_samples": int(sample_counts[subgroup]),
                "n_pairs": 0,
                "max_violation": np.nan,
                "avg_violation": np.nan,
                "violation_rate": np.nan,
            })
            continue

        v = violations[mask]
        records.append({
            "attribute": attribute_name,
            "subgroup": subgroup,
            "n_samples": int(sample_counts[subgroup]),
            "n_pairs": n_pairs,
            "max_violation": float(v.max()),
            "avg_violation": float(v.mean()),
            "violation_rate": float((v > 0).mean()),
        })

    table = pd.DataFrame(records)
    return table.sort_values(["violation_rate", "n_pairs"], ascending=[False, False], na_position="last")

for attribute in analysis_attrs:
    print(f"\nWithin-group IF by {attribute}")
    print(within_group_if_table_from_labels(sensitive_test[attribute], attribute).to_string(index=False))

intersection_name = "marital-status x sex"
intersection_labels = (
    sensitive_test["marital-status"].fillna("<missing>").astype(str)
    + " | "
    + sensitive_test["sex"].fillna("<missing>").astype(str)
)
print("\nWithin-group IF by (marital-status, sex) intersection")
print(
    within_group_if_table_from_labels(
        intersection_labels,
        intersection_name,
    ).to_string(index=False)
)



Within-group IF by marital-status
     attribute              subgroup  n_samples  n_pairs  max_violation  avg_violation  violation_rate
marital-status              Divorced        883     2640       0.396209       0.024547        0.318561
marital-status    Married-civ-spouse       2977    25074       0.773675       0.028579        0.301508
marital-status         Never-married       2181    18553       0.601545       0.012216        0.152482
marital-status             Separated        199      144       0.403079       0.011038        0.145833
marital-status Married-spouse-absent         86       14       0.038749       0.005536        0.142857
marital-status               Widowed        182      358       0.273141       0.004430        0.069832
marital-status     Married-AF-spouse          5        0            NaN            NaN             NaN

Within-group IF by sex
attribute subgroup  n_samples  n_pairs  max_violation  avg_violation  violation_rate
      sex     Male       4355   

## Cause of Unfairness (Data, Minimal)
Start with data-only evidence: representation skew, label-rate skew, and local-pair coverage. This adds diagnostics only and does not retrain the model.


> **Data Skew Diagnostics (Minimal Next Step)**
> Prints subgroup/intersection counts, `>50K` rates, and local IF pair coverage (`n_pairs`) for `marital-status`, `sex`, and `marital-status x sex`.


In [8]:
# Minimal data-skew diagnostics (no retraining)
required = [
    "data", "target", "y_test", "X_sensitivefeatures_test",
    "rows", "cols", "violations",
    "within_group_if_table_from_labels", "analysis_attrs",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError("Run the earlier cells first. Missing: " + ", ".join(missing))

positive_label = ">50K"
intersection_name = globals().get("intersection_name", "marital-status x sex")

# Full dataset: representation + label skew by subgroup / intersection
full_diag = data[analysis_attrs + [target]].copy()
full_diag[intersection_name] = (
    full_diag[analysis_attrs[0]].fillna("<missing>").astype(str)
    + " | "
    + full_diag[analysis_attrs[1]].fillna("<missing>").astype(str)
)
full_diag["__positive__"] = (full_diag[target] == positive_label).astype(int)

# Test split version (useful for comparing with IF outputs computed on test data)
test_diag = X_sensitivefeatures_test[analysis_attrs].reset_index(drop=True).copy()
test_diag[target] = y_test.reset_index(drop=True)
test_diag[intersection_name] = (
    test_diag[analysis_attrs[0]].fillna("<missing>").astype(str)
    + " | "
    + test_diag[analysis_attrs[1]].fillna("<missing>").astype(str)
)
test_diag["__positive__"] = (test_diag[target] == positive_label).astype(int)

for label, frame in [("full dataset", full_diag), ("test split", test_diag)]:
    print(f"\nData skew diagnostics ({label})")
    for attr in analysis_attrs + [intersection_name]:
        summary = (
            frame.groupby(attr)
            .agg(
                n_samples=(attr, "size"),
                positive_rate=("__positive__", "mean"),
            )
            .sort_values(["n_samples"], ascending=False)
        )
        summary["positive_rate"] = summary["positive_rate"].round(4)
        print(f"\nCounts and positive rate by {attr}")
        print(summary.to_string())

# Pair coverage in the already-computed local IF graph (within-group only)
print("\nLocal-pair coverage in the current IF audit (test split)")
for attr in analysis_attrs:
    tbl = within_group_if_table_from_labels(sensitive_test[attr], attr)
    print(f"\nPair coverage summary for {attr}")
    print(tbl[["subgroup", "n_samples", "n_pairs", "violation_rate"]].to_string(index=False))

intersection_labels = (
    sensitive_test[analysis_attrs[0]].fillna("<missing>").astype(str)
    + " | "
    + sensitive_test[analysis_attrs[1]].fillna("<missing>").astype(str)
)
intersection_tbl = within_group_if_table_from_labels(intersection_labels, intersection_name)
print(f"\nPair coverage summary for {intersection_name}")
print(intersection_tbl[["subgroup", "n_samples", "n_pairs", "violation_rate"]].to_string(index=False))



Data skew diagnostics (full dataset)

Counts and positive rate by marital-status
                       n_samples  positive_rate
marital-status                                 
Married-civ-spouse         14976         0.4468
Never-married              10683         0.0460
Divorced                    4443         0.1042
Separated                   1025         0.0644
Widowed                      993         0.0856
Married-spouse-absent        418         0.0813
Married-AF-spouse             23         0.4348

Counts and positive rate by sex
        n_samples  positive_rate
sex                             
Male        21790         0.3057
Female      10771         0.1095

Counts and positive rate by marital-status x sex
                                n_samples  positive_rate
marital-status x sex                                    
Married-civ-spouse | Male           13319         0.4458
Never-married | Male                 5916         0.0549
Never-married | Female               4767  

## Cause of Unfairness (Data Intervention, Minimal)
A single controlled data intervention: rebalance the training set by `sex` using undersampling, keep the test set and method fixed, and compare fairness/performance to baseline.


> **Sex-Balanced Undersampling (3 Seeds)**
> Repeats the same train-only undersampling experiment with 3 random seeds to check whether the fairness change is consistent rather than a one-off sample effect.


In [9]:
# Minimal data intervention: sex-balanced undersampling (3 seeds)
import time
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.neighbors import NearestNeighbors

required = [
    "classifierPipeline", "X_train", "X_test", "y_train", "y_test",
    "data", "target", "analysis_attrs", "X_sensitivefeatures_test",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise NameError("Run the earlier cells first. Missing: " + ", ".join(missing))

if9_positive_label = ">50K"
if9_k_local = 15
if9_seeds = [42, 43, 44]
if9_intersection_name = "marital-status x sex"


def if9_within_group_if_table_from_labels(labels, rows_, cols_, violations_, attribute_name):
    labels = labels.fillna("<missing>").astype(str)
    labels_arr = labels.to_numpy()
    group_i = labels_arr[rows_]
    group_j = labels_arr[cols_]
    within_mask = group_i == group_j
    sample_counts = labels.value_counts().to_dict()

    records = []
    for subgroup in sorted(sample_counts):
        mask = within_mask & (group_i == subgroup)
        n_pairs = int(mask.sum())
        if n_pairs == 0:
            records.append({
                "attribute": attribute_name,
                "subgroup": subgroup,
                "n_samples": int(sample_counts[subgroup]),
                "n_pairs": 0,
                "max_violation": np.nan,
                "avg_violation": np.nan,
                "violation_rate": np.nan,
            })
            continue

        v = violations_[mask]
        records.append({
            "attribute": attribute_name,
            "subgroup": subgroup,
            "n_samples": int(sample_counts[subgroup]),
            "n_pairs": n_pairs,
            "max_violation": float(v.max()),
            "avg_violation": float(v.mean()),
            "violation_rate": float((v > 0).mean()),
        })

    table = pd.DataFrame(records)
    return table.sort_values(["violation_rate", "n_pairs"], ascending=[False, False], na_position="last")


def if9_evaluate_model(fitted_model):
    # Performance metrics (same test set)
    y_pred_local = fitted_model.predict(X_test)
    performance = {
        "accuracy": float(accuracy_score(y_test, y_pred_local)),
        "macro_f1": float(f1_score(y_test, y_pred_local, average="macro")),
        "positive_recall": float(recall_score(y_test, y_pred_local, pos_label=if9_positive_label)),
    }

    # Global IF (same local-pair protocol as earlier cell)
    proba = fitted_model.predict_proba(X_test)
    pos_idx = list(fitted_model.classes_).index(if9_positive_label)
    p_local = proba[:, pos_idx]

    X_test_preprocessed_local = fitted_model.named_steps["preprocess"].transform(X_test)
    nn_local = NearestNeighbors(n_neighbors=min(if9_k_local + 1, len(X_test)), metric="euclidean")
    nn_local.fit(X_test_preprocessed_local)
    distances_local, indices_local = nn_local.kneighbors(X_test_preprocessed_local)

    rows_local = np.repeat(np.arange(len(X_test)), distances_local.shape[1] - 1)
    cols_local = indices_local[:, 1:].reshape(-1)
    raw_d_local = distances_local[:, 1:].reshape(-1)

    scale_local = float(raw_d_local.max()) if raw_d_local.size else 1.0
    if scale_local == 0:
        scale_local = 1.0
    d_local = np.clip(raw_d_local / scale_local, 0.0, 1.0)

    delta_p_local = np.abs(p_local[rows_local] - p_local[cols_local])
    violations_local = np.maximum(0.0, delta_p_local - d_local)

    sensitive_test_local = X_sensitivefeatures_test[analysis_attrs].reset_index(drop=True).copy()
    sex_tbl = if9_within_group_if_table_from_labels(
        sensitive_test_local["sex"], rows_local, cols_local, violations_local, "sex"
    )

    intersection_labels_local = (
        sensitive_test_local["marital-status"].fillna("<missing>").astype(str)
        + " | "
        + sensitive_test_local["sex"].fillna("<missing>").astype(str)
    )
    intersection_tbl = if9_within_group_if_table_from_labels(
        intersection_labels_local, rows_local, cols_local, violations_local, if9_intersection_name
    )

    return {
        "performance": performance,
        "global_if": {
            "evaluated_pairs": int(len(violations_local)),
            "max_violation": float(violations_local.max()) if violations_local.size else 0.0,
            "avg_violation": float(violations_local.mean()) if violations_local.size else 0.0,
            "violation_rate": float((violations_local > 0).mean()) if violations_local.size else 0.0,
        },
        "sex_table": sex_tbl,
        "intersection_table": intersection_tbl,
    }


def if9_get_violation_rate(table, subgroup_name):
    row = table[table["subgroup"] == subgroup_name]
    if row.empty:
        return np.nan
    return float(row.iloc[0]["violation_rate"])


def if9_sex_balanced_train_index(seed):
    train_sex = data.loc[X_train.index, "sex"].astype(str)
    counts = train_sex.value_counts()
    if len(counts) != 2:
        raise ValueError(f"Expected 2 sex groups, got: {counts.to_dict()}")

    minority_n = int(counts.min())
    rng = np.random.default_rng(seed)
    selected = []
    for group_name in sorted(counts.index.tolist()):
        group_index = train_sex[train_sex == group_name].index.to_numpy()
        if len(group_index) > minority_n:
            chosen = rng.choice(group_index, size=minority_n, replace=False)
        else:
            chosen = group_index
        selected.append(chosen)

    selected_index = pd.Index(np.concatenate(selected)).sort_values()
    return selected_index, counts.to_dict(), minority_n


# Baseline (current fitted model) for comparison
if9_baseline_eval = if9_evaluate_model(classifierPipeline)
if9_baseline_male = if9_get_violation_rate(if9_baseline_eval["sex_table"], "Male")
if9_baseline_female = if9_get_violation_rate(if9_baseline_eval["sex_table"], "Female")

if9_rows = [{
    "experiment": "baseline_current_train",
    "seed": np.nan,
    "train_rows": int(len(X_train)),
    "accuracy": if9_baseline_eval["performance"]["accuracy"],
    "macro_f1": if9_baseline_eval["performance"]["macro_f1"],
    "positive_recall": if9_baseline_eval["performance"]["positive_recall"],
    "global_if_violation_rate": if9_baseline_eval["global_if"]["violation_rate"],
    "global_if_avg_violation": if9_baseline_eval["global_if"]["avg_violation"],
    "sex_if_male": if9_baseline_male,
    "sex_if_female": if9_baseline_female,
    "sex_if_gap_male_minus_female": if9_baseline_male - if9_baseline_female,
    "notes": "Original training data",
}]

print("Sex-balanced undersampling experiment (3 seeds)")
print("- Baseline train rows:", len(X_train))

for seed in if9_seeds:
    train_index_bal, sex_counts_train, target_n = if9_sex_balanced_train_index(seed)
    print(f"\nSeed {seed}: sex counts in original train = {sex_counts_train}; selected per sex = {target_n}; selected rows = {len(train_index_bal)}")

    pipe_bal = clone(classifierPipeline)
    start = time.time()
    pipe_bal.fit(X_train.loc[train_index_bal], y_train.loc[train_index_bal])
    elapsed = time.time() - start

    eval_bal = if9_evaluate_model(pipe_bal)
    male_rate = if9_get_violation_rate(eval_bal["sex_table"], "Male")
    female_rate = if9_get_violation_rate(eval_bal["sex_table"], "Female")

    if9_rows.append({
        "experiment": "sex_balanced_undersample",
        "seed": int(seed),
        "train_rows": int(len(train_index_bal)),
        "accuracy": eval_bal["performance"]["accuracy"],
        "macro_f1": eval_bal["performance"]["macro_f1"],
        "positive_recall": eval_bal["performance"]["positive_recall"],
        "global_if_violation_rate": eval_bal["global_if"]["violation_rate"],
        "global_if_avg_violation": eval_bal["global_if"]["avg_violation"],
        "sex_if_male": male_rate,
        "sex_if_female": female_rate,
        "sex_if_gap_male_minus_female": male_rate - female_rate,
        "fit_time_sec": float(elapsed),
        "notes": "Sex-balanced undersampling (train only)",
    })

if9_results = pd.DataFrame(if9_rows)
baseline_row = if9_results.loc[if9_results["experiment"] == "baseline_current_train"].iloc[0]
for metric in [
    "accuracy", "macro_f1", "positive_recall",
    "global_if_violation_rate", "global_if_avg_violation",
    "sex_if_gap_male_minus_female",
]:
    if9_results[f"delta_{metric}"] = if9_results[metric] - float(baseline_row[metric])

print("\nBaseline vs 3 sex-balanced runs")
print(
    if9_results[
        [
            "experiment", "seed", "train_rows",
            "accuracy", "macro_f1", "positive_recall",
            "global_if_violation_rate", "global_if_avg_violation",
            "sex_if_male", "sex_if_female", "sex_if_gap_male_minus_female",
            "delta_accuracy", "delta_macro_f1", "delta_positive_recall",
            "delta_global_if_violation_rate", "delta_global_if_avg_violation",
            "delta_sex_if_gap_male_minus_female",
        ]
    ].round(6).to_string(index=False)
)

balanced_only = if9_results[if9_results["experiment"] == "sex_balanced_undersample"].copy()
if not balanced_only.empty:
    summary = balanced_only[
        [
            "accuracy", "macro_f1", "positive_recall",
            "global_if_violation_rate", "global_if_avg_violation",
            "sex_if_gap_male_minus_female",
            "delta_accuracy", "delta_macro_f1", "delta_positive_recall",
            "delta_global_if_violation_rate", "delta_global_if_avg_violation",
            "delta_sex_if_gap_male_minus_female",
        ]
    ].agg(["mean", "std"]).round(6)
    print("\nSex-balanced runs (3 seeds) summary: mean / std")
    print(summary.to_string())


Sex-balanced undersampling experiment (3 seeds)
- Baseline train rows: 26048

Seed 42: sex counts in original train = {'Male': 17435, 'Female': 8613}; selected per sex = 8613; selected rows = 17226

Seed 43: sex counts in original train = {'Male': 17435, 'Female': 8613}; selected per sex = 8613; selected rows = 17226

Seed 44: sex counts in original train = {'Male': 17435, 'Female': 8613}; selected per sex = 8613; selected rows = 17226

Baseline vs 3 sex-balanced runs
              experiment  seed  train_rows  accuracy  macro_f1  positive_recall  global_if_violation_rate  global_if_avg_violation  sex_if_male  sex_if_female  sex_if_gap_male_minus_female  delta_accuracy  delta_macro_f1  delta_positive_recall  delta_global_if_violation_rate  delta_global_if_avg_violation  delta_sex_if_gap_male_minus_female
  baseline_current_train   NaN       26048  0.817288  0.714579         0.451531                  0.260648                 0.022848     0.282337       0.223326                      0.05